<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
This notebook adapts material from the <a href="https://docs.ros.org/en/jazzy/">ROS 2 documentation</a> (<a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>) and from the Duckietown <em>ROS Basics</em> and <em>ROS 2 Middleware</em> learning experiences.
</span>

# Inspect the Duckiebot's ROS 2 Graph

The previous notebook connected a local publisher to its topic, delivery settings and received messages, and measured arrival rate and latency. On a Duckiebot, the same tools inspect the data supplied by the sensor drivers and consumed by the actuator drivers.

Before an application can consume that data, it needs the topic's resolved name, message type and compatible delivery settings. How can those details be established from the running system? This notebook follows the camera images and the wheel encoder ticks through that investigation, first from a terminal and then from this notebook, which can display the data live.

**This notebook needs a robot**: a real Duckiebot or a virtual one in the Duckiematrix, with its ROS 2 stack running. The README explains how to create a virtual robot (`dts duckiebot virtual create/start`), start the Duckiematrix (`dts code start_matrix`) and bring the stack up:

```bash
dts stack up -R ROBOTNAME ros2/duckiebot
dts stack up -R ROBOTNAME ros2-core/default
```

## Connect a terminal to the robot

Open a terminal and point it at the robot's zenoh router (notebook 04):

```bash
export ZENOH_CONFIG_OVERRIDE='mode="client";connect/endpoints=["tcp/ROBOTNAME.local:7447"]'
ros2 daemon stop
```

Confirm the configuration:

```bash
printenv ZENOH_CONFIG_OVERRIDE
```

```shell
mode="client";connect/endpoints=["tcp/ROBOTNAME.local:7447"]
```

(the terminal is a zenoh *client* of the robot's router). Every terminal you open for this notebook needs the same `export`.

## Discover the robot's nodes and topics

The familiar commands reveal the running nodes and typed topics. The graph takes a few seconds to sync through the router, so allow time for discovery:

```bash
ros2 node list --no-daemon --spin-time 5
```

The listing should contain the nodes of the two stacks:

```shell
/ROBOTNAME/camera_node
/ROBOTNAME/car_cmd_switch_node
/ROBOTNAME/kinematics_node
/ROBOTNAME/led_driver_node
/ROBOTNAME/left_wheel_encoder_driver_node
/ROBOTNAME/right_wheel_encoder_driver_node
/ROBOTNAME/tof_driver_node
/ROBOTNAME/velocity_to_pose_node
/ROBOTNAME/wheels_driver_node
```

Other nodes may appear (a joystick mapper, for instance), and a node may be missing if its hardware is absent. The drivers come from the `ros2/duckiebot` stack; `kinematics_node`, `car_cmd_switch_node` and `velocity_to_pose_node` from `ros2-core/default`. Every node lives in the namespace `/ROBOTNAME`, as discussed in notebook 04.

```bash
ros2 topic list -t --no-daemon --spin-time 5
```

The topic listing should include entries such as:

```shell
/ROBOTNAME/camera_node/camera_info [sensor_msgs/msg/CameraInfo]
/ROBOTNAME/camera_node/image/compressed [sensor_msgs/msg/CompressedImage]
/ROBOTNAME/car_cmd_switch_node/cmd [duckietown_msgs/msg/Twist2DStamped]
/ROBOTNAME/left_wheel_encoder_driver_node/tick [duckietown_msgs/msg/WheelEncoderStamped]
/ROBOTNAME/right_wheel_encoder_driver_node/tick [duckietown_msgs/msg/WheelEncoderStamped]
/ROBOTNAME/wheels_driver_node/wheels_cmd [duckietown_msgs/msg/WheelsCmdStamped]
/ROBOTNAME/wheels_driver_node/wheels_cmd_executed [duckietown_msgs/msg/WheelsCmdStamped]
```

These excerpts show only part of the graph. Compare them with Figure 1 of notebook 02: the sensor topics are at the top of the data path, `car_cmd_switch_node/cmd` is where your node will publish, and `wheels_driver_node/wheels_cmd` is what the kinematics node produces for the motors.

## Identify the camera publisher

The image topic offers a compact way of following the sensor data path:

```bash
ros2 topic info --verbose /ROBOTNAME/camera_node/image/compressed
```

The output should identify `sensor_msgs/msg/CompressedImage` and its publisher. Relevant excerpts are:

```shell
Type: sensor_msgs/msg/CompressedImage
Publisher count: 1

Node name: camera_node
Node namespace: /ROBOTNAME
Endpoint type: PUBLISHER
QoS profile:
  Reliability: RELIABLE
  History (Depth): KEEP_LAST (1)
  Durability: VOLATILE
```

The node name and namespace identify `/ROBOTNAME/camera_node` as the publisher. Subscription counts may vary because the dashboard or another observer can also consume this topic.

The history depth of `1` is a deliberate choice for a sensor stream: the driver keeps only the latest frame for a subscriber that falls behind, rather than queuing old images. Compare with the depth of `10` of the text publisher of notebook 07. An application that processes images should expect to receive the most recent frame, not every frame.

The same inspection on `/ROBOTNAME/left_wheel_encoder_driver_node/tick` shows the encoder driver with the same policies.

## Receive sensor messages

With the topic and type established, a temporary observer can request one message. Encoder ticks are small:

```bash
ros2 topic echo --once --timeout 10 /ROBOTNAME/left_wheel_encoder_driver_node/tick
```

```shell
header:
  stamp:
    sec: 1793450123
    nanosec: 456789000
  frame_id: ROBOTNAME/left_wheel_axis
data: 1532
resolution: 135
type: 1
---
```

`data` is the tick count since the driver started (it increases when the wheel turns forward and decreases backwards), and `resolution` says that `135` ticks make one wheel revolution; `type: 1` is `ENCODER_TYPE_INCREMENTAL` (notebook 03). Values depend on your robot and should not match this example exactly. Turn the wheel by hand on a real robot, or drive the robot (notebook 09), and the count changes.

Images are large; an echo would fill the terminal with thousands of bytes. The `--no-arr` option hides array fields:

```bash
ros2 topic echo --once --timeout 10 --no-arr /ROBOTNAME/camera_node/image/compressed
```

```shell
header:
  stamp:
    sec: 1793450124
    nanosec: 12345000
  frame_id: camera_color_optical_frame
format: jpeg
data: '<sequence type: uint8, length: 48213>'
---
```

A JPEG image of about 48 kB arrived. Measure the rates:

```bash
ros2 topic hz /ROBOTNAME/camera_node/image/compressed
ros2 topic hz /ROBOTNAME/left_wheel_encoder_driver_node/tick
```

The camera publishes at around 20 to 30 frames per second, depending on the robot; the encoder publishes while the wheel turns (so the second measurement may stay silent on a robot that is standing still). Stop each measurement with `Ctrl-C`.

If a command returns without a message after the waiting period, inspect the publisher's availability (`ros2 node list`) and the router connection again. A listed interface can remain visible even when the expected data is not arriving.

## The same graph, from the notebook

Everything above can be done from a notebook cell. The cells below create a ROS 2 node inside this notebook, connected to the robot's router, and use it to list the graph and display live data. Fill in your robot's name first.

In [ ]:
ROBOT = "ROBOTNAME"   # <-- the name of your (real or virtual) Duckiebot

In [ ]:
from lx_ros2_tools.session import RosSession
from lx_ros2_tools.panel import ImageView, TopicPlot

ros = RosSession(ROBOT)   # connects to tcp/ROBOT.local:7447 and spins in a background thread
ros

`RosSession` is a node like any other: it appears as `/lx_notebook_...` in `ros2 node list`. Its `nodes()` and `topics()` methods are the notebook equivalent of `ros2 node list` and `ros2 topic list -t`. The graph needs a few seconds to sync, so wait a little (or run the cell twice) if the lists are short.

In [ ]:
import time
time.sleep(5)  # let discovery settle
print("\n".join(sorted(ros.nodes())))

In [ ]:
for name, types in sorted(ros.topics().items()):
    if name.startswith(f"/{ROBOT}/"):
        print(f"{name}  {types}")

### Wait for one message

`ros.wait_for_message(topic, type)` is the notebook equivalent of `ros2 topic echo --once`. Topic names without a leading `/` are relative to the robot's namespace, so `camera_node/camera_info` means `/ROBOT/camera_node/camera_info`.

In [ ]:
from sensor_msgs.msg import CameraInfo
from duckietown_msgs.msg import WheelEncoderStamped

info = ros.wait_for_message("camera_node/camera_info", CameraInfo, timeout=10)
print(f"camera: {info.width}x{info.height} pixels, frame '{info.header.frame_id}'")

tick = ros.wait_for_message("left_wheel_encoder_driver_node/tick", WheelEncoderStamped, timeout=10, sensor=True)
print(f"left encoder: {tick.data} ticks, {tick.resolution} ticks per revolution")

`sensor=True` subscribes with the *sensor data* QoS profile (best effort, small queue), the profile to use for images and encoder ticks; notebook 10 explains why.

### See what the Duckiebot sees

`ImageView` subscribes to a `CompressedImage` topic and shows the frames in the notebook (throttled to a few frames per second).

In [ ]:
camera = ImageView(ros, "camera_node/image/compressed")

In the Duckiematrix, move the duckie in front of the robot (keys `w`, `a`, `s`, `d` in the simulator window) and watch the image change. On a real robot, wave at the camera.

### Plot the encoders

`TopicPlot` plots numeric fields of a topic over time. Here, the tick count of each wheel:

In [ ]:
left = TopicPlot(ros, "left_wheel_encoder_driver_node/tick", "duckietown_msgs/msg/WheelEncoderStamped", ["data"],
                 sensor=True, title="left wheel encoder ticks")
right = TopicPlot(ros, "right_wheel_encoder_driver_node/tick", "duckietown_msgs/msg/WheelEncoderStamped", ["data"],
                  sensor=True, title="right wheel encoder ticks")

The plots update as messages arrive. On a real robot, turn a wheel by hand and watch the count rise (forward) or fall (backward). In the Duckiematrix the wheels only turn when commanded; keep this notebook open and come back after starting `drive_node` in notebook 09, or publish a few commands from the next cell (the robot moves for one second!).

In [ ]:
from duckietown_msgs.msg import Twist2DStamped

cmd = Twist2DStamped()
for _ in range(10):                      # 10 commands at 10 Hz = 1 s of motion
    cmd.header.stamp = ros.node.get_clock().now().to_msg()
    cmd.v, cmd.omega = 0.2, 0.0          # 0.2 m/s straight ahead
    ros.publish("car_cmd_switch_node/cmd", cmd)
    time.sleep(0.1)
cmd.v = 0.0
ros.publish("car_cmd_switch_node/cmd", cmd)   # stop

Both encoder counts should have increased by roughly the same amount, and the camera image changed. If the robot did not move, check which topic its `kinematics_node` reads with `ros2 node info /ROBOTNAME/kinematics_node` (see the README's troubleshooting section).

### Clean up

Panels keep their subscriptions until they are closed. Close them when you are done (the session itself can stay open; it is reused by the next cells you run, and a notebook restart ends it):

In [ ]:
camera.close()
left.close()
right.close()

Inspection from a terminal connected to the robot's router identified the camera and encoder topics, their publishing nodes, message definitions and delivery policies. Receiving samples provided information about the sensors, and the same observations were repeated from the notebook with live views. The next notebook writes a node that publishes commands on this graph.

## Further reading

The [Duckietown manual](https://docs.duckietown.com/ente/) describes the Duckiebot's software stack. The [ROS 2 QoS documentation](https://docs.ros.org/en/jazzy/Concepts/Intermediate/About-Quality-of-Service-Settings.html) explains delivery-policy compatibility.

Let's move to [the next notebook about publishers](../notebooks/09_publishers.ipynb).